#### **Task: Self-Attention with Trainable Weights**
### Sentence

##### **"The student reads a book."**

In [6]:
### input Embeddings
import torch
inputs = torch.tensor([
    [0.2, 0.5, 0.1, 0.3],  # The
    [0.9, 0.8, 0.4, 0.7],  # student
    [0.6, 0.3, 0.8, 0.5],  # reads
    [0.1, 0.2, 0.1, 0.2],  # a
    [0.7, 0.6, 0.9, 0.8]   # book
])

In [7]:
### trainable Weights

##1.Query

W_query = torch.tensor([
 [0.5, 0.2, 0.4],
 [0.3, 0.7, 0.1],
 [0.6, 0.4, 0.8],
 [0.2, 0.5, 0.3]
])

In [8]:
### 2. Key
W_key = torch.tensor([
 [0.4, 0.6, 0.2],
 [0.7, 0.3, 0.5],
 [0.1, 0.8, 0.4],
 [0.5, 0.2, 0.7]
])

In [9]:
### 3. value
W_value = torch.tensor([
 [0.6, 0.1, 0.5],
 [0.2, 0.8, 0.3],
 [0.7, 0.4, 0.6],
 [0.3, 0.5, 0.9]
])

In [10]:
### Query vector = inputs X Query Weight
queries = inputs @ W_query
queries

tensor([[0.3700, 0.5800, 0.3000],
        [1.0700, 1.2500, 0.9700],
        [0.9700, 0.9000, 1.0600],
        [0.2100, 0.3000, 0.2000],
        [1.2300, 1.3200, 1.3000]])

In [11]:
### Key Vector = inputs X Key Weight
keys = inputs @ W_key
keys

tensor([[0.5900, 0.4100, 0.5400],
        [1.3100, 1.2400, 1.2300],
        [0.7800, 1.1900, 0.9400],
        [0.2900, 0.2400, 0.3000],
        [1.1900, 1.4800, 1.3600]])

In [12]:
### Value Vector = inputs X Value Weights
values = inputs @ W_value
values

tensor([[0.3800, 0.6100, 0.5800],
        [1.1900, 1.2400, 1.5600],
        [1.1300, 0.8700, 1.3200],
        [0.2300, 0.3100, 0.3500],
        [1.4100, 1.3100, 1.7900]])

In [17]:
### Let's try with the word reads, attention scores. 
query_2 = queries[2]
query_2

tensor([0.9700, 0.9000, 1.0600])

#### **Attention Scores**

#### ***How strongly one token pay Attention to another token.*** 

In [19]:
keys

tensor([[0.5900, 0.4100, 0.5400],
        [1.3100, 1.2400, 1.2300],
        [0.7800, 1.1900, 0.9400],
        [0.2900, 0.2400, 0.3000],
        [1.1900, 1.4800, 1.3600]])

In [20]:
## Attention scores = Query X key

attn_score_2 = query_2 @ keys.T
attn_score_2

tensor([1.5137, 3.6905, 2.8240, 0.8153, 3.9279])

In [24]:
## normalize the attention score
d_k = inputs.shape[-1]

attn_weights_2 = torch.softmax(attn_score_2/d_k**0.5,dim = -1)
attn_weights_2

tensor([0.1006, 0.2986, 0.1936, 0.0709, 0.3363])

In [25]:
attn_weights_2.sum()

tensor(1.)

In [26]:
## let's implement to all queries and keys
attn_scores = queries @ keys.T
attn_scores

tensor([[0.6181, 1.5729, 1.2608, 0.3365, 1.7067],
        [1.6676, 4.1448, 3.2339, 0.9013, 4.4425],
        [1.5137, 3.6905, 2.8240, 0.8153, 3.9279],
        [0.3549, 0.8931, 0.7088, 0.1929, 0.9659],
        [1.9689, 4.8471, 3.7522, 1.0635, 5.1853]])

In [27]:
## attention weights
attn_weights = torch.softmax(attn_scores/d_k**0.5,dim = -1)
attn_weights

tensor([[0.1519, 0.2449, 0.2095, 0.1320, 0.2618],
        [0.0883, 0.3047, 0.1932, 0.0602, 0.3536],
        [0.1006, 0.2986, 0.1936, 0.0709, 0.3363],
        [0.1730, 0.2264, 0.2064, 0.1595, 0.2348],
        [0.0753, 0.3174, 0.1836, 0.0479, 0.3759]])

In [29]:
attn_weights_2

tensor([0.1006, 0.2986, 0.1936, 0.0709, 0.3363])

In [28]:
values

tensor([[0.3800, 0.6100, 0.5800],
        [1.1900, 1.2400, 1.5600],
        [1.1300, 0.8700, 1.3200],
        [0.2300, 0.3100, 0.3500],
        [1.4100, 1.3100, 1.7900]])

In [30]:
## context vector =  attention weights X values

context_vector_2 = attn_weights_2 @ values
context_vector_2

tensor([1.1028, 1.0626, 1.4065])

In [31]:
## apply to all
context_vectors = attn_weights @ values
context_vectors

tensor([[0.9853, 0.9624, 1.2614],
        [1.1269, 1.0817, 1.4356],
        [1.1028, 1.0626, 1.4065],
        [0.9360, 0.9228, 1.2020],
        [1.1548, 1.1065, 1.4707]])

In [34]:
d_in = 4
d_out = 3

In [35]:
import torch.nn as nn

class SimpleAttention_V1(nn.Module):

    def __init__(self,d_in,s_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in,d_out),requires_grad = False)
        self.W_key = nn.Parameter(torch.rand(d_in,d_out),requires_grad = False)
        self.W_value = nn.Parameter(torch.rand(d_in,d_out),requires_grad = False)

    def forward(self,x):
        keys = x@self.W_key
        queries = x@self.W_query 
        values = x@self.W_value


        attention_scores = queries@keys.T
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5,dim= -1)
        context_vectors = attention_weights@values

        return context_vectors

In [36]:
simple_attn = SimpleAttention_V1(d_in,d_out)
simple_attn(inputs)

tensor([[0.4698, 0.7620, 0.2514],
        [0.5471, 0.8811, 0.2857],
        [0.5404, 0.8713, 0.2835],
        [0.4369, 0.7096, 0.2341],
        [0.5563, 0.8953, 0.2892]])